# 08. Generation logprob inspector

This notebook inspects how token probabilities evolve during generation using **PyTorch** and **Hugging Face Transformers** directly. The goal is to make logits, logprobs, and token boundaries visible step by step.

## What this notebook shows

You can configure:

- the prompt
- the model name
- the number of generated tokens
- the top K candidate tokens to inspect at each step

For each generated step, the notebook displays the top K candidate tokens with:

- token text
- token ID
- logit
- logprob
- probability

It also renders the generated output with alternating background colors so token boundaries are easy to see.

In [ ]:

import math
import html
import torch
import pandas as pd
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForCausalLM


## Configuration

Change these values and rerun the notebook to inspect different generation behaviors.

In [ ]:

MODEL_NAME = 'sshleifer/tiny-gpt2'
PROMPT = 'The future of language models is'
MAX_NEW_TOKENS = 8
TOP_K = 8
TEMPERATURE = 1.0
DEVICE = 'cpu'


## Load model and tokenizer

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()


## Helper functions

We will:

- render tokens with alternating background colors
- inspect the top K candidates from the next-token distribution
- generate one token at a time so each step is visible

In [ ]:

TOKEN_COLORS = ['#eef6ff', '#fff5e6']

def render_tokens(token_ids, tokenizer):
    pieces = []
    for i, tok_id in enumerate(token_ids):
        token_text = tokenizer.decode([tok_id], clean_up_tokenization_spaces=False)
        safe = html.escape(token_text).replace(' ', '&nbsp;').replace('
', '<br>')
        color = TOKEN_COLORS[i % len(TOKEN_COLORS)]
        pieces.append(
            f"<span style='background:{color}; padding:4px 6px; margin:2px; border-radius:4px; display:inline-block; font-family:monospace;'>"
            f"{safe}<span style='color:#666; font-size:0.8em;'> [{tok_id}]</span></span>"
        )
    return HTML(''.join(pieces))


def top_k_table(logits, top_k, tokenizer):
    probs = torch.softmax(logits, dim=-1)
    logprobs = torch.log_softmax(logits, dim=-1)
    top = torch.topk(probs, k=top_k)

    rows = []
    for prob, idx in zip(top.values.tolist(), top.indices.tolist()):
        rows.append({
            'token_text': tokenizer.decode([idx], clean_up_tokenization_spaces=False),
            'token_id': idx,
            'logit': float(logits[idx]),
            'logprob': float(logprobs[idx]),
            'probability': float(prob),
        })
    return pd.DataFrame(rows)


## Step-by-step generation

At each step we:

1. run the model on the current sequence
2. inspect the next-token logits
3. show the top K candidate tokens
4. greedily pick the most likely next token (or temperature-scaled argmax)
5. append it and continue

In [ ]:

inputs = tokenizer(PROMPT, return_tensors='pt')
input_ids = inputs['input_ids'].to(DEVICE)
attention_mask = inputs['attention_mask'].to(DEVICE)

generated = input_ids.clone()
step_results = []

for step in range(MAX_NEW_TOKENS):
    with torch.no_grad():
        outputs = model(input_ids=generated, attention_mask=torch.ones_like(generated))

    next_token_logits = outputs.logits[0, -1] / TEMPERATURE
    table = top_k_table(next_token_logits, TOP_K, tokenizer)

    probs = torch.softmax(next_token_logits, dim=-1)
    chosen_token_id = int(torch.argmax(probs).item())
    chosen_logprob = float(torch.log_softmax(next_token_logits, dim=-1)[chosen_token_id].item())
    chosen_prob = float(probs[chosen_token_id].item())
    chosen_text = tokenizer.decode([chosen_token_id], clean_up_tokenization_spaces=False)

    step_results.append({
        'step': step + 1,
        'chosen_token_id': chosen_token_id,
        'chosen_token_text': chosen_text,
        'chosen_logprob': chosen_logprob,
        'chosen_probability': chosen_prob,
        'table': table,
    })

    next_token_tensor = torch.tensor([[chosen_token_id]], device=DEVICE)
    generated = torch.cat([generated, next_token_tensor], dim=1)


## Generated tokens with boundaries

The display below highlights token boundaries with alternating colors, which helps show how tokenization splits text.

In [ ]:

prompt_token_ids = input_ids[0].tolist()
generated_token_ids = generated[0].tolist()

print('Prompt tokens:')
display(render_tokens(prompt_token_ids, tokenizer))

print('Prompt + generated tokens:')
display(render_tokens(generated_token_ids, tokenizer))

print('Decoded text:')
print(tokenizer.decode(generated_token_ids, clean_up_tokenization_spaces=False))


## Per-step top K candidates

Each table below shows the model's local next-token distribution at one generation step.

In [ ]:

for item in step_results:
    print(f"
Step {item['step']}")
    print('Chosen token:', repr(item['chosen_token_text']), 'id=', item['chosen_token_id'])
    print('Chosen logprob:', item['chosen_logprob'])
    print('Chosen probability:', item['chosen_probability'])
    display(item['table'])


## Notes

- **Logits** are the raw scores before normalization.
- **Logprobs** are the log of the normalized probabilities.
- **Probabilities** are easier to read directly, but logprobs are often more stable and additive across steps.
- A token with the highest probability at one step is not guaranteed to remain dominant later, because the context changes after each generated token.